# 11. 오일 표 보완과 전체 임베딩 실험

## 실행 순서
1. 10번 노트북의 전체 청킹 자료를 메모리에서 다시 만듭니다.
2. PDF 43쪽(매뉴얼 38쪽)의 표를 종류·용량·추천 사양·각주 관계로 보완합니다.
3. 모든 검색 조각을 로컬 한국어 모델로 숫자 벡터로 바꿉니다.
4. 같은 질문으로 의미 검색과 검색어를 결합한 검색을 비교합니다.

원본 43쪽을 화면으로 대조했습니다. 리어 디퍼런셜과 트랜스퍼 케이스는 추천 사양 셀을 공유합니다.
엔진 오일 4.8 ℓ에는 일반적인 교체 시 주입량이라는 각주가 있으므로 용량만 단독으로 답하면 안 됩니다.

새 코드에는 한글 주석을 넣었습니다. 셀을 위에서부터 실행하세요.
전체 실행에는 시간이 걸릴 수 있습니다. 진행률은 처리한 조각 수로 표시합니다.
벡터는 메모리에 두며 데이터 파일이나 Supabase에 저장하지 않습니다.


In [1]:
from pathlib import Path
import json
import io
import contextlib
import re
from copy import deepcopy
from collections import Counter, defaultdict
import time

project_folder = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
previous_path = project_folder / "notebooks" / "10_paddle_source_review.ipynb"
previous_notebook = json.loads(previous_path.read_text(encoding="utf-8"))
previous = {}
with contextlib.redirect_stdout(io.StringIO()):
    for index, cell in enumerate(previous_notebook["cells"]):
        if cell["cell_type"] == "code":
            exec(compile("".join(cell["source"]), f"10:cell_{index}", "exec"), previous)

page_inventory = previous["page_inventory"]
prepare_text = previous["prepare_text"]
make_reviewed_chunks = previous["make_reviewed_chunks"]
token_count = previous["draft"]["token_count"]
token_budget = previous["token_budget"]
model_name = previous["draft"]["type_example"]["model_name"]
original_parents = previous["reviewed_parent_records"]
full_parent_records = deepcopy(original_parents)
print("불러온 원문 묶음:", len(full_parent_records))
print("모델:", model_name, "| 입력 한도:", token_budget)


불러온 원문 묶음: 526
모델: jhgan/ko-sroberta-multitask-mrl | 입력 한도: 128


## 1. 표를 보완하고 각주·주의사항 연결하기

표의 일곱 행을 화면 기준으로 옮깁니다. 숫자와 단위는 원문대로 유지합니다.
공유하는 사양은 두 행에 각각 기록하고, 원본에서 같은 셀이었음을 표시합니다.
하단 각주와 주의사항은 별도 부모 기록으로 남겨 표와 연결합니다.


In [2]:
oil_matches = [
    p for p in full_parent_records
    if p["metadata"]["title"] == "추천 오일 및 용량"
    and p["metadata"]["pdf_page_number"] == 43
]
if len(oil_matches) != 1:
    raise ValueError("43쪽 원문 묶음의 경계를 다시 확인하세요.")
oil_original = oil_matches[0]
assert oil_original["metadata"]["source_pages"] == [43]
assert oil_original["metadata"]["manual_page_number"] == 38
assert len(oil_original["metadata"]["source_spans"]) == 1
original_span = oil_original["metadata"]["source_spans"][0]
body = page_inventory[43]["body_text"]
footnote_start = body.index("엔진 오일 용량은 일반적인 오일 교체 시 주입되는 용량 기준입니다.")
table_raw = body[original_span["start"]:footnote_start]
notes_raw = body[footnote_start:original_span["end"]]

shared_specification = "하이포이드 기어 오일 API GL-5, SAE 75W/85(SK HCT-5 기어 오일 75W/85 or 상당품)"
oil_rows = [
    {"kind": "연료", "capacity": "67 ℓ", "specification": "무연 휘발유", "footnote_ids": []},
    {"kind": "엔진 오일", "capacity": "4.8 ℓ",
     "specification": "SAE 0W-20 API SN PLUS/SP 또는 ILSAC GF-6", "footnote_ids": ["*1", "*2"]},
    {"kind": "자동 변속기 오일", "capacity": "6.0 ℓ",
     "specification": "SK ATF SP4M-1, MICHANG ATF SP4M-1, S-OIL ATF SP4M-1, Hyundai Genuine ATF SP4M-1",
     "footnote_ids": []},
    {"kind": "엔진 냉각수", "capacity": "8.9 ℓ",
     "specification": "알루미늄 라디에이터용 인산염계 에틸렌 글리콜 부동액과 물 혼합액",
     "footnote_ids": []},
    {"kind": "브레이크액", "capacity": "필요량", "specification": "DOT-4", "footnote_ids": ["*3"]},
    {"kind": "리어 디퍼런셜 오일 (HTRAC)", "capacity": "0.53 ~ 0.63 ℓ",
     "specification": shared_specification, "footnote_ids": ["*4"], "shared_specification_cell": "htrac_gears"},
    {"kind": "트랜스퍼 케이스 오일 (HTRAC)", "capacity": "0.62 ~ 0.68 ℓ",
     "specification": shared_specification, "footnote_ids": ["*4"], "shared_specification_cell": "htrac_gears"},
]

# 표 셀 안에서 줄이 바뀐 영문명도 화면에서 같은 문구임을 확인했습니다.
compact_raw = re.sub(r"\s+", "", table_raw)
for row in oil_rows:
    for field in ["kind", "capacity", "specification"]:
        assert re.sub(r"\s+", "", row[field]) in compact_raw

note_markers = [
    ("*1", "엔진 오일 용량은 일반적인"),
    ("*2", "API SN PLUS(또는 그 이상)"),
    ("*3", "차량의 제동 성능 및 ABS/ESC"),
    ("*4", "교체 주기와는 별도로"),
]
caution_start = notes_raw.index("઱੄")
footnotes = {}
for index, (note_id, marker) in enumerate(note_markers):
    start = notes_raw.index(marker)
    end = notes_raw.index(note_markers[index + 1][1]) if index + 1 < len(note_markers) else caution_start
    footnotes[note_id] = prepare_text(notes_raw[start:end])
caution_text = prepare_text(notes_raw[caution_start:])

oil_table_id = oil_original["metadata"]["record_id"]
oil_notes_id = "reviewed_oil_notes_43"
table_content = "\n".join([
    "추천 오일 및 용량",
    *[
        row["kind"] + " | 용량: " + row["capacity"] + " | 추천 사양: " + row["specification"]
        + (" | 각주: " + ", ".join(row["footnote_ids"]) if row["footnote_ids"] else "")
        for row in oil_rows
    ],
])
oil_table_record = {
    "content": table_content, "raw_text": table_raw, "image_parts": [],
    "metadata": deepcopy(oil_original["metadata"]),
}
oil_table_record["metadata"].update({
    "content_type": "structured_table", "table_rows": oil_rows,
    "source_spans": [{"pdf_page_number": 43, "start": original_span["start"], "end": footnote_start}],
    "verification_status": "visually_reviewed_source", "review_flags": [],
    "table_structure_status": "visually_verified",
    "review_method": "43쪽 전체 화면의 표 셀·공유 사양·각주 번호 대조",
    "required_context_record_ids": [oil_notes_id],
})
oil_notes_record = {
    "content": "추천 오일 및 용량 — 각주 및 주의사항\n"
        + "\n".join(note_id + ": " + text for note_id, text in footnotes.items())
        + "\n" + caution_text,
    "raw_text": notes_raw, "image_parts": [],
    "metadata": {
        "record_id": oil_notes_id, "title": "추천 오일 및 용량 — 각주 및 주의사항",
        "pdf_page_number": 43, "manual_page_number": 38, "source_pages": [43],
        "chapter_id": oil_original["metadata"].get("chapter_id"),
        "content_type": "instruction_text",
        "source_spans": [{"pdf_page_number": 43, "start": footnote_start, "end": original_span["end"]}],
        "verification_status": "visually_reviewed_source", "review_flags": [],
        "review_method": "43쪽 전체 화면의 각주·주의사항 대조",
        "footnotes": footnotes, "required_context_record_ids": [oil_table_id],
    },
}
assert table_raw + notes_raw == oil_original["raw_text"]
assert "[주의]" in oil_notes_record["content"]
assert "엔진 오일 첨가제를 사용하지 마십시오." in oil_notes_record["content"]
assert "'F' 선 이상 주입하지" in oil_notes_record["content"]
full_parent_records = [
    record for parent in full_parent_records
    for record in ([oil_table_record, oil_notes_record]
        if parent["metadata"]["record_id"] == oil_table_id else [parent])
]
print("표 행:", len(oil_rows), "| 연결 각주:", list(footnotes))
print("공유 사양:", oil_rows[-2]["specification"] == oil_rows[-1]["specification"])
print("엔진 오일 행:", oil_rows[1])
print("연결된 *1 각주:", footnotes["*1"])


표 행: 7 | 연결 각주: ['*1', '*2', '*3', '*4']
공유 사양: True
엔진 오일 행: {'kind': '엔진 오일', 'capacity': '4.8 ℓ', 'specification': 'SAE 0W-20 API SN PLUS/SP 또는 ILSAC GF-6', 'footnote_ids': ['*1', '*2']}
연결된 *1 각주: 엔진 오일 용량은 일반적인 오일 교체 시 주입되는 용량 기준입니다.


## 2. 전체 조각과 출처 관계 확인

표의 한 행은 자르지 않습니다. 한 행이 입력 한도를 넘으면 원본을 다시 확인하도록 멈춥니다.
본문은 문장 단위로 나누고, 검색된 작은 조각에서 부모 원문을 찾아갈 수 있게 ID를 유지합니다.


In [3]:
# 앞 노트북이 만든 일반 조각·표 조각 분할 함수를 재사용합니다.
full_search_chunks = [
    chunk for parent in full_parent_records for chunk in make_reviewed_chunks(parent)
]
parent_lookup = {parent["metadata"]["record_id"]: parent for parent in full_parent_records}
chunk_lookup = {chunk["metadata"]["record_id"]: chunk for chunk in full_search_chunks}

assert len(parent_lookup) == len(full_parent_records)
assert len(chunk_lookup) == len(full_search_chunks)
assert all(token_count(c["content"]) == c["metadata"]["token_count"] <= token_budget
    for c in full_search_chunks)

# 새 구간은 표와 각주로 나뉘었지만 이전 구간 전체를 정확히 덮어야 합니다.
new_spans = [
    span for record in [oil_table_record, oil_notes_record]
    for span in record["metadata"]["source_spans"]
]
assert new_spans[0]["start"] == original_span["start"]
assert new_spans[0]["end"] == new_spans[1]["start"]
assert new_spans[1]["end"] == original_span["end"]
for parent in original_parents:
    if parent["metadata"]["record_id"] != oil_table_id:
        assert parent == parent_lookup[parent["metadata"]["record_id"]]

groups = defaultdict(list)
for index, chunk in enumerate(full_search_chunks):
    groups[chunk["metadata"]["parent_record_id"]].append(index)
for parent_id, parent in parent_lookup.items():
    cursor = 0
    for index in groups[parent_id]:
        chunk = full_search_chunks[index]
        assert chunk["metadata"]["parent_content_start"] == cursor
        cursor = chunk["metadata"]["parent_content_end"]
    assert cursor == len(parent["content"])
for row in table_content.splitlines()[1:]:
    assert any(row in full_search_chunks[i]["content"] for i in groups[oil_table_id])

print("원문 묶음:", len(full_parent_records), "| 검색 조각:", len(full_search_chunks))
print("상태별 기록 수:", dict(Counter(p["metadata"]["verification_status"] for p in full_parent_records)))
print("표 행·원문 구간·부모 연결:", "확인 완료")


원문 묶음: 527 | 검색 조각: 2213
상태별 기록 수: {'sample_verified': 14, 'auto_draft_needs_review': 507, 'visually_reviewed_source': 6}
표 행·원문 구간·부모 연결: 확인 완료


## 3. 로컬 임베딩 모델 준비

임베딩은 글을 768개의 숫자로 바꾸는 작업입니다.
`normalize_embeddings=True`는 벡터 길이를 1로 맞춰 유사도를 비교하기 쉽게 합니다.
현재 PC에서는 CPU로 계산하고 이미 내려받은 모델만 읽습니다.


In [4]:
import os
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# CPU를 과도하게 점유하지 않도록 계산 스레드는 최대 네 개로 제한합니다.
torch.set_num_threads(min(4, os.cpu_count() or 1))
embedding_model = SentenceTransformer(model_name, device="cpu", local_files_only=True)
assert embedding_model.max_seq_length == token_budget
vector_dimension = embedding_model.get_embedding_dimension()
assert vector_dimension == 768
batch_size = 16
print("실행 환경: CPU | 스레드:", torch.get_num_threads(), "| 묶음 크기:", batch_size)
print("벡터 차원:", vector_dimension, "| 최대 입력:", embedding_model.max_seq_length)


실행 환경: CPU | 스레드: 4 | 묶음 크기: 16
벡터 차원: 768 | 최대 입력: 128


## 4. 전체 임베딩 실행

16개씩 처리하고 64개가 완료될 때마다 진행 건수와 시간을 출력합니다.
오류가 생기면 실패한 구간을 표시하고 멈춥니다. 조각을 건너뛰어 벡터 순서가 달라지는 것을 막습니다.
완료 후 `chunk_vectors[i]`는 `full_search_chunks[i]`의 벡터입니다.


In [5]:
embedding_started = time.perf_counter()
total_chunks = len(full_search_chunks)
chunk_vectors = np.empty((total_chunks, vector_dimension), dtype=np.float32)
embedding_failures = []
embedded_count = 0
for start in range(0, total_chunks, batch_size):
    end = min(start + batch_size, total_chunks)
    try:
        vectors = embedding_model.encode(
            [c["content"] for c in full_search_chunks[start:end]],
            batch_size=batch_size, convert_to_numpy=True,
            normalize_embeddings=True, show_progress_bar=False,
        )
        assert vectors.shape == (end - start, vector_dimension)
        assert np.isfinite(vectors).all()
        chunk_vectors[start:end] = vectors
        embedded_count = end
    except Exception as error:
        embedding_failures.append({"start": start, "end": end, "error": str(error)})
        print("실패한 조각 구간:", start, "~", end, "| 오류:", str(error), flush=True)
        raise
    if end % 64 == 0 or end == total_chunks:
        elapsed = time.perf_counter() - embedding_started
        remaining = (total_chunks - end) * elapsed / end
        print(f"임베딩 {end}/{total_chunks} | 오류 {len(embedding_failures)} | "
              f"경과 {elapsed:.1f}초 | 예상 남은 시간 {remaining:.1f}초", flush=True)

embedding_elapsed = time.perf_counter() - embedding_started
assert embedded_count == total_chunks and not embedding_failures
assert chunk_vectors.shape == (total_chunks, 768)
assert np.isfinite(chunk_vectors).all()
assert np.allclose(np.linalg.norm(chunk_vectors, axis=1), 1.0, atol=1e-5)
print("임베딩 완료:", chunk_vectors.shape, "| 총 시간:", round(embedding_elapsed, 1), "초")
print("벡터 메모리:", round(chunk_vectors.nbytes / 1024**2, 2), "MiB")


임베딩 64/2213 | 오류 0 | 경과 5.0초 | 예상 남은 시간 167.6초
임베딩 128/2213 | 오류 0 | 경과 9.9초 | 예상 남은 시간 161.2초
임베딩 192/2213 | 오류 0 | 경과 14.9초 | 예상 남은 시간 157.3초
임베딩 256/2213 | 오류 0 | 경과 20.0초 | 예상 남은 시간 152.7초
임베딩 320/2213 | 오류 0 | 경과 25.1초 | 예상 남은 시간 148.3초
임베딩 384/2213 | 오류 0 | 경과 30.3초 | 예상 남은 시간 144.2초
임베딩 448/2213 | 오류 0 | 경과 35.4초 | 예상 남은 시간 139.3초
임베딩 512/2213 | 오류 0 | 경과 40.2초 | 예상 남은 시간 133.6초
임베딩 576/2213 | 오류 0 | 경과 46.3초 | 예상 남은 시간 131.5초
임베딩 640/2213 | 오류 0 | 경과 52.2초 | 예상 남은 시간 128.2초
임베딩 704/2213 | 오류 0 | 경과 57.6초 | 예상 남은 시간 123.4초
임베딩 768/2213 | 오류 0 | 경과 63.4초 | 예상 남은 시간 119.2초
임베딩 832/2213 | 오류 0 | 경과 69.8초 | 예상 남은 시간 115.8초
임베딩 896/2213 | 오류 0 | 경과 75.3초 | 예상 남은 시간 110.7초
임베딩 960/2213 | 오류 0 | 경과 81.4초 | 예상 남은 시간 106.2초
임베딩 1024/2213 | 오류 0 | 경과 86.9초 | 예상 남은 시간 100.9초
임베딩 1088/2213 | 오류 0 | 경과 92.5초 | 예상 남은 시간 95.6초
임베딩 1152/2213 | 오류 0 | 경과 98.5초 | 예상 남은 시간 90.7초
임베딩 1216/2213 | 오류 0 | 경과 106.5초 | 예상 남은 시간 87.3초
임베딩 1280/2213 | 오류 0 | 경과 115.4초 | 예상 남은 시간 84.1초
임베딩 1344/2213 | 오류 0

## 5. 의미 검색과 결합 검색

의미 검색은 질문 벡터와 비슷한 청크를 찾습니다.
결합 검색은 의미 점수 50%, 글자 2~4개 단위의 검색어 일치 점수 50%를 사용합니다.
글자 색인은 희소 행렬(값이 있는 곳만 저장하는 배열)로 만들어 메모리 사용을 줄입니다.
표의 `+`, `-`도 색인에 남깁니다. 평가 결과에 맞춰 가중치를 조절하지 않습니다.

두 방식 모두 부모 기록은 중복 없이 반환하고, 점수는 정답 확률로 해석하지 않습니다.


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

parent_ids = list(parent_lookup)
parent_positions = {parent_id: index for index, parent_id in enumerate(parent_ids)}

def normalize_keyword(text):
    """한글·영문·숫자와 표 조작 기호를 남겨 띄어쓰기 차이를 줄입니다."""
    return "".join(re.findall(r"[가-힣A-Za-z0-9+\-]+", text)).lower()

lexical_texts = []
for parent_id in parent_ids:
    parent = parent_lookup[parent_id]
    items = parent["metadata"].get("navigation_items")
    # 안내도 참조 쪽수를 부품 이름으로 오인하지 않도록 이름만 색인합니다.
    text = "\n".join(item["name"] for item in items) if items else parent["content"]
    lexical_texts.append(normalize_keyword(text))
lexical_index = TfidfVectorizer(analyzer="char", ngram_range=(2, 4), dtype=np.float32)
lexical_matrix = lexical_index.fit_transform(lexical_texts)
semantic_weight = 0.5

def rank_candidates(question, question_vector, method="semantic", top_k=3):
    """가장 잘 맞는 자식 조각의 점수로 부모 원문 후보를 정렬합니다."""
    if method not in {"semantic", "hybrid"}:
        raise ValueError("검색 방식은 semantic 또는 hybrid입니다.")
    if not 1 <= top_k <= len(parent_ids):
        raise ValueError("검색 결과 수를 확인하세요.")
    chunk_scores = chunk_vectors @ question_vector
    keyword_scores = np.zeros(len(parent_ids), dtype=np.float32)
    if method == "hybrid":
        query_terms = lexical_index.transform([normalize_keyword(question)])
        keyword_scores = (lexical_matrix @ query_terms.T).toarray().ravel()
    hits = []
    for parent_id, indices in groups.items():
        best_index = max(indices, key=lambda index: float(chunk_scores[index]))
        semantic_score = float(chunk_scores[best_index])
        keyword_score = float(keyword_scores[parent_positions[parent_id]])
        score = semantic_score if method == "semantic" else (
            semantic_weight * semantic_score + (1 - semantic_weight) * keyword_score
        )
        hits.append({
            "score": score, "semantic_score": semantic_score, "keyword_score": keyword_score,
            "matched_chunk": full_search_chunks[best_index],
            "parent_record": parent_lookup[parent_id],
        })
    hits.sort(key=lambda hit: hit["score"], reverse=True)
    return hits[:top_k]

def full_manual_search(question, method="hybrid", top_k=3):
    """질문 길이를 확인한 뒤 전체 청크에서 출처가 담긴 검색 후보를 찾습니다."""
    if not question.strip() or token_count(question) > token_budget:
        raise ValueError("질문을 비우지 말고 입력 한도 안에서 짧게 작성하세요.")
    vector = embedding_model.encode(
        question, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False
    )
    return rank_candidates(question, vector, method=method, top_k=top_k)

def read_context(parent_id):
    """부모 원문과 필수 연결 기록을 읽되, 아직 검토 중인 자료는 상태를 함께 남깁니다."""
    pending, seen, context = [parent_id], set(), []
    while pending:
        record_id = pending.pop(0)
        if record_id in seen:
            continue
        seen.add(record_id)
        record = parent_lookup[record_id]
        context.append(record)
        pending.extend(record["metadata"].get("required_context_record_ids", []))
    return context

print("글자 색인:", lexical_matrix.shape, "| 실제 저장 값:", lexical_matrix.nnz)
print("의미 점수 비중:", semantic_weight)


글자 색인: (527, 211160) | 실제 저장 값: 565051
의미 점수 비중: 0.5


## 6. 기대 출처를 미리 정한 질문으로 검색 확인

앞 실험의 20개 질문과 이번 표·패들 쉬프트 질문 10개를 사용합니다.
기대 ID는 검색하기 전에 정합니다. 동일 표가 398쪽과 402쪽에 있으므로 두 출처 모두 인정합니다.
상위 1개·3개에 기대 자료가 있는지 확인하며, 챗봇 답변의 정확도를 평가하는 것은 아닙니다.


In [7]:
import ast

# 06번의 질문 목록만 읽습니다. 소규모 임베딩 실험을 다시 실행하지 않습니다.
baseline_notebook = json.loads((project_folder / "notebooks" /
    "06_embedding_parent_retrieval.ipynb").read_text(encoding="utf-8"))
baseline_questions = None
for cell in baseline_notebook["cells"]:
    if cell["cell_type"] != "code":
        continue
    tree = ast.parse("".join(cell["source"]))
    for statement in tree.body:
        if isinstance(statement, ast.Assign) and any(
            isinstance(target, ast.Name) and target.id == "evaluation_questions"
            for target in statement.targets
        ):
            baseline_questions = ast.literal_eval(statement.value)
assert baseline_questions is not None

paddle_id = previous["paddle_id"]
table_398_id = previous["table_398_id"]
table_402_id = previous["table_402_id"]
limit_402_id = previous["limit_402_id"]
evaluation_cases = [
    {"question": question, "expected_ids": [expected_id], "group": "기존 확인 질문"}
    for question, expected_id in baseline_questions
] + [
    {"question": "엔진 오일의 용량과 추천 사양을 알려줘.", "expected_ids": [oil_table_id]},
    {"question": "자동 변속기 오일 용량과 추천 사양은?", "expected_ids": [oil_table_id]},
    {"question": "리어 디퍼런셜 오일과 트랜스퍼 케이스 오일 용량은?", "expected_ids": [oil_table_id]},
    {"question": "브레이크액의 추천 규격과 관련 각주를 찾아줘.", "expected_ids": [oil_table_id, oil_notes_id]},
    {"question": "엔진 오일 4.8리터는 일반적인 교체 시 주입량인가요?", "expected_ids": [oil_notes_id]},
    {"question": "엔진 오일 첨가제를 사용하면 안 되는 이유는?", "expected_ids": [oil_notes_id]},
    {"question": "ECO 모드에서 패들 쉬프트 플러스 레버를 당기면?", "expected_ids": [table_398_id, table_402_id]},
    {"question": "SPORT 모드에서 패들 쉬프트 레버는 어떤 기능인가요?", "expected_ids": [table_398_id, table_402_id, paddle_id]},
    {"question": "패들 쉬프트를 동시에 당기면 변속되나요?", "expected_ids": [paddle_id]},
    {"question": "패들 쉬프트로 회생 제동량을 변경할 수 없는 상황은?", "expected_ids": [limit_402_id]},
]
for case in evaluation_cases:
    case.setdefault("group", "이번 원본 확인 질문")
    assert token_count(case["question"]) <= token_budget
    assert set(case["expected_ids"]) <= set(parent_lookup)

query_vectors = embedding_model.encode(
    [case["question"] for case in evaluation_cases],
    batch_size=batch_size, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False,
)
evaluation_results = []
for case, vector in zip(evaluation_cases, query_vectors):
    row = dict(case)
    expected = set(case["expected_ids"])
    for method in ["semantic", "hybrid"]:
        hits = rank_candidates(case["question"], vector, method=method, top_k=3)
        ids = [hit["parent_record"]["metadata"]["record_id"] for hit in hits]
        row[method + "_ids"] = ids
        row[method + "_top1"] = ids[0] in expected
        row[method + "_top3"] = bool(expected.intersection(ids))
    evaluation_results.append(row)

for group in ["기존 확인 질문", "이번 원본 확인 질문", "전체"]:
    rows = [r for r in evaluation_results if r["group"] == group] if group != "전체" else evaluation_results
    print("\n", group, "| 질문:", len(rows))
    for method in ["semantic", "hybrid"]:
        print(method, "| 첫 결과:", sum(r[method + "_top1"] for r in rows),
              "| 상위 세 결과:", sum(r[method + "_top3"] for r in rows))
print("\n첫 결과가 기대 출처와 다른 질문:")
for row in evaluation_results:
    if not row["hybrid_top1"]:
        first_id = row["hybrid_ids"][0]
        first = parent_lookup[first_id]
        print("-", row["question"])
        print("  첫 결과:", first["metadata"]["title"], "| PDF", first["metadata"]["source_pages"],
              "| 검토 상태:", first["metadata"]["verification_status"])
        print("  기대 출처:", row["expected_ids"], "| 상위3:", row["hybrid_ids"])



 기존 확인 질문 | 질문: 20
semantic | 첫 결과: 5 | 상위 세 결과: 6
hybrid | 첫 결과: 6 | 상위 세 결과: 12

 이번 원본 확인 질문 | 질문: 10
semantic | 첫 결과: 8 | 상위 세 결과: 10
hybrid | 첫 결과: 8 | 상위 세 결과: 9

 전체 | 질문: 30
semantic | 첫 결과: 13 | 상위 세 결과: 16
hybrid | 첫 결과: 14 | 상위 세 결과: 21

첫 결과가 기대 출처와 다른 질문:
- 실외 미러를 접는 버튼은 차량 내부 안내도의 어디에 있나요?
  첫 결과: 실외 미러 | PDF [246, 247, 248, 249] | 검토 상태: auto_draft_needs_review
  기대 출처: ['overview_33'] | 상위3: ['auto_topic_150', 'auto_topic_24', 'auto_topic_149']
- 후드 열림 레버의 위치를 보여주는 안내도를 찾아줘.
  첫 결과: 후드 여는 방법 | PDF [266] | 검토 상태: auto_draft_needs_review
  기대 출처: ['overview_33'] | 상위3: ['auto_topic_163', 'auto_topic_162', 'overview_33']
- 지문 인증 시스템은 차량 내부 안내도에서 몇 번인가요?
  첫 결과: 지문 인증 시스템 | PDF [211] | 검토 상태: auto_draft_needs_review
  기대 출처: ['overview_33'] | 상위3: ['auto_topic_114', 'auto_topic_116', 'auto_topic_117']
- 유리창 잠금 버튼과 전자식 차일드 락 버튼 위치를 알려줘.
  첫 결과: 전자식 차일드 락 | PDF [221, 222] | 검토 상태: auto_draft_needs_review
  기대 출처: ['overview_33'] | 상위3: ['auto_topic_126', 'auto_topic_352', 'a

## 7. 출처·각주·그림 연결 예시와 자료 밖 질문

오일 표가 반환되면 각주·주의사항도 함께 가져옵니다.
검토가 필요한 자동 초안이 검색되면 그 상태를 숨기지 않습니다.
자료 밖 질문에도 유사한 문서가 반환될 수 있으므로 검색 점수만으로 답변하면 안 됩니다.


In [8]:
demo_question = "엔진 오일의 용량과 추천 사양을 알려줘."
demo_hits = full_manual_search(demo_question, top_k=3)
print("질문:", demo_question)
for hit in demo_hits:
    parent = hit["parent_record"]
    print("후보:", parent["metadata"]["title"], "| PDF", parent["metadata"]["source_pages"],
          "| 점수:", round(hit["score"], 4), "| 상태:", parent["metadata"]["verification_status"])
oil_context = read_context(oil_table_id)
assert oil_notes_id in {r["metadata"]["record_id"] for r in oil_context}
print("오일 표에서 함께 읽는 원문:", [r["metadata"]["title"] for r in oil_context])

# 그림은 검색 청크가 아니라 연결된 부모 기록에서 가져옵니다.
paddle_parent = parent_lookup[paddle_id]
part = paddle_parent["image_parts"][0]
resolved_image = previous["reader"].pages[part["pdf_page_number"] - 1].images[part["pdf_image_key"]]
assert resolved_image.name == part["name"]
assert tuple(resolved_image.image.size) == tuple(part["size"])
print("패들 쉬프트 실제 그림 연결:", part["pdf_page_number"], part["name"], resolved_image.image.size)

outside_questions = ["오늘 서울 날씨와 내일 강수확률은?", "싼타페 중고차의 현재 시세는 얼마인가요?"]
for question in outside_questions:
    hit = full_manual_search(question, top_k=1)[0]
    parent = hit["parent_record"]
    print("\n자료 밖 질문:", question)
    print("그래도 반환된 후보:", parent["metadata"]["title"], "| 점수:", round(hit["score"], 4))
print("\n정답 판정·자동 답변:", "아직 구현하지 않음")
print("전체 임베딩:", embedded_count, "/", total_chunks, "| 실패:", len(embedding_failures))
print("데이터 파일·벡터 파일·Supabase 저장:", "실행하지 않음")


질문: 엔진 오일의 용량과 추천 사양을 알려줘.
후보: 추천 오일 및 용량 — 각주 및 주의사항 | PDF [43] | 점수: 0.5254 | 상태: visually_reviewed_source
후보: 추천 오일 및 용량 | PDF [43] | 점수: 0.5203 | 상태: visually_reviewed_source
후보: 엔진 오일 | PDF [705] | 점수: 0.4689 | 상태: auto_draft_needs_review
오일 표에서 함께 읽는 원문: ['추천 오일 및 용량', '추천 오일 및 용량 — 각주 및 주의사항']
패들 쉬프트 실제 그림 연결: 397 I2.jpg (687, 438)

자료 밖 질문: 오늘 서울 날씨와 내일 강수확률은?
그래도 반환된 후보: 폭설 시 행동요령 | 점수: 0.1983

자료 밖 질문: 싼타페 중고차의 현재 시세는 얼마인가요?
그래도 반환된 후보: 중고차 구입/디지털 키 관련 차량 정비 | 점수: 0.2969

정답 판정·자동 답변: 아직 구현하지 않음
전체 임베딩: 2213 / 2213 | 실패: 0
데이터 파일·벡터 파일·Supabase 저장: 실행하지 않음


## 다음 작업

검색 순위가 기대와 다른 질문은 원문 경계·질문 표현·표와 그림의 연결을 살펴봅니다.
현재 다른 자동 초안은 검토 상태를 유지합니다. 임베딩 완료는 원문 검토 완료를 뜻하지 않습니다.

검색 실험을 정리한 뒤 노트북의 준비·청킹·임베딩·검색을 .py 함수로 옮기고,
사용자가 이해할 수 있는 실행 방법을 정리합니다. DB 저장은 이후 개인 영역의 구조를 확인하고 진행합니다.


## 이번 실행 결과

- 원문 묶음 **527개**, 검색 조각 **2,213개**를 전체 임베딩했습니다.
- 벡터 배열 **(2213, 768)**, 계산 시간 **206.7초**, 벡터 배열 메모리 **6.48 MiB**입니다. 이 메모리 수치는 모델과 기타 자료의 메모리를 포함하지 않습니다.
- 처리 누락·실패 0개, 모든 입력 128토큰 이내, 벡터 숫자가 유한하며 길이가 1인지 확인했습니다.
- 상태: 이전 확인 예제 14개, 원본 화면 대조 6개, 검토가 필요한 자동 초안 507개입니다.

### 출처 검색 비교

| 질문 묶음 | 질문 수 | 의미 검색 상위1 | 의미 검색 상위3 | 결합 검색 상위1 | 결합 검색 상위3 |
|---|---:|---:|---:|---:|---:|
| 기존 확인 질문 | 20 | 5 | 6 | 6 | 12 |
| 이번 표·패들 질문 | 10 | 8 | 10 | 8 | 9 |
| 합계 | 30 | 13 | 16 | 14 | 21 |

이 표는 **검색 전에 지정한 원문 ID**가 포함됐는지를 계산한 것이며 실제 답변 정확도가 아닙니다.
전체 자료에서는 같은 내용의 상세 설명이 다른 출처에 있을 수 있습니다. 이번 비교에서는 그 대체 출처를 모두 검토하지 않았습니다.

### 확인된 검색 과제

1. 안내도·위치·번호를 묻는 질문에 상세 기능 설명이 먼저 검색되는 경우가 많았습니다. 질문 목적에 맞는 자료 유형 선택을 다음 실험에서 다룹니다.
2. 브레이크액 규격·각주 질문은 상세 브레이크액 문서가 표보다 먼저 검색돼 이번 결합 검색의 상위3에 지정 출처가 없었습니다. 이번 새 질문 10개에서는 의미 검색 상위3에 모두 지정 출처가 있었으므로 결합 가중치가 모든 질문을 개선했다고 볼 수 없습니다.
3. 엔진 오일 용량·사양 질문의 첫 결과는 같은 43쪽 각주 기록이었습니다. 연결을 읽으면 표도 가져옵니다. 이번 지표는 첫 ID만 비교해 이 경우를 첫 결과 불일치로 셉니다.
4. 자료 밖 날씨·시세 질문에도 후보가 반환됐습니다. 검색 점수는 답변 가능 여부를 판단하는 근거로 충분하지 않습니다.

### 저장과 이어가기

벡터 파일이나 Supabase 저장은 하지 않았습니다. 이번 실행의 결과 요약만 노트북에 보관합니다.
새 커널이나 별도 실행에서 벡터를 사용하려면 임베딩 셀을 다시 실행해야 합니다.

다음은 검색 유형 선택과 연결된 근거 읽기를 보완한 뒤, 이 실험 코드를 개인 .py 모듈로 정리하는 단계입니다.
